In [ ]:
import os
import sqlite3

import hydra
import pandas as pd
from hydra import compose, initialize

### Load configs

In [ ]:
# Initialize Hydra with the config directory (relative to notebooks/)
if not hydra.core.global_hydra.GlobalHydra.instance().is_initialized():
    initialize(config_path="../config", version_base=None)

# Load the evaluation.yaml configuration
cfg_eval = compose(config_name="evaluation")
cfg_dataset = compose(config_name="dataset_creation")
print(cfg_dataset)
print(cfg_eval)

In [ ]:
def get_data_from_db(path_db: str, table: str = "CoRal_recording") -> pd.DataFrame:
    """Gets the data from the database.

    Args:
        path_db (str): The path to the database file.
        table (str): Name of table to load.

    Returns:
        pd.DataFrame: The data from the database.
    """
    connection = sqlite3.connect(path_db)
    read_aloud_data = pd.read_sql_query(
        sql="SELECT * FROM {}".format(table), con=connection
    )
    return read_aloud_data

In [ ]:
dir_transcriptions = os.path.join(cfg_dataset.dir_data_raw, "transcriptions")

lst_transcriptions = os.listdir(dir_transcriptions)
lst_transcriptions = [s for s in lst_transcriptions if ".ass" in s]
lst_transcriptions

df_transcriptions = pd.DataFrame({"filename": lst_transcriptions})


def get_recid(filename: str):
    """Extract the conversation ID from a filename.

    Args:
        filename (str): The name of the file.

    Returns:
        str: The conversation ID.
    """
    filename = filename.lstrip("@")
    if filename.startswith("r"):
        recid = filename.split("_")[0]
    else:
        recid = filename.rstrip(".ass")
    return recid


df_transcriptions["id_conversation"] = df_transcriptions.filename.apply(
    lambda x: get_recid(x)
)

df_transcriptions

In [ ]:
df_conv_meta_new = get_data_from_db(cfg_dataset.metadata_database_path, "Conversations")
df_conv_meta_new["datetime_start"] = pd.to_datetime(
    df_conv_meta_new["datetime_start"], format="%Y-%m-%d %H:%M:%S"
)
df_conv_meta_new["datetime_end"] = pd.to_datetime(
    df_conv_meta_new["datetime_end"], format="%Y-%m-%d %H:%M:%S"
)

df_conv_meta_new = df_conv_meta_new.drop(
    columns=["location_roomdim", "noise_level", "noise_type", "transcription"]
)
df_conv_meta_new

In [ ]:
df_speakers = get_data_from_db(cfg_dataset.metadata_database_path, "Speakers")
df_speakers.drop(columns=["education", "occupation"])

In [ ]:
df_speakers["dialect"] = (
    df_speakers["dialect"]
    .map(cfg_eval.sub_dialect_to_dialect)
    .fillna(df_speakers["dialect"])
)
df_speakers["dialect"].value_counts()

In [ ]:
import gspread
from google.auth import default

# Authenticate and create the client


creds, _ = default()
gc = gspread.authorize(creds)


# Open the Google Sheet (replace with your sheet name or URL)
sheet = gc.open(
    "https://docs.google.com/spreadsheets/d/1TOiPNSNGkUkAIm8QyfUkP0uyImLknPtexg7uAzXU3w0/edit?gid=1592810641#gid=1592810641"
).sheet1

# Get all records and convert to DataFrame
data = sheet.get_all_records()
df_google_sheet = pd.DataFrame(data)

df_google_sheet